# Guía L2 — El partido que se acumula · **Sesión 2: fabricar**

**Cálculo Integral — Laboratorio computacional** · Universidad Nacional de Colombia, Sede Manizales

> En la sesión 1 recibieron el momentum hecho y lo acumularon. Hoy reciben la materia prima, las 4.386 acciones de la final con sus coordenadas, y fabrican **su propio** momentum. Después lo comparan con el del curso y con el partido que el mundo vio.

**Antes de empezar:** copia propia del cuaderno (`Archivo → Guardar una copia en Drive`), la tarea puente 1 lista y a la mano su respuesta sobre "los goles que pesan raro", porque hoy la van a usar. Las celdas marcadas con ✏️ son de ustedes. Las marcadas con 🗣️ son para escribir conclusiones del equipo, con sus palabras.

---
## ⏸️ Momento 1 — Predecir

P4 y P5 están en la guía. **En papel, antes de ejecutar nada**, con foto para el informe.

- **P4.** Si en la receta multiplican por 10 el peso de los remates, ¿cambia el *signo* del dominio neto? ¿Y el minuto del máximo de $D$?
- **P5.** ¿Puede la gráfica de momentum *anticipar* un gol, o solo lo celebra después?

---
## 1 · Conocer el evento a evento

Cada fila de `eventos_final2022.csv` es una acción del partido. Columnas:

- `periodo`, `minuto`, `segundo`: el reloj oficial (periodos 1 y 2, el tiempo reglamentario; 3 y 4, la prórroga).
- `t_corrido`: el minuto corrido de la sesión 1, ahora con decimales.
- `equipo` y `tipo`: quién hizo la acción y qué fue, con los nombres de StatsBomb en inglés (`Pass`, `Carry`, `Shot`, `Dribble`, `Foul Won`…).
- `x`, `y`: dónde **empieza** la acción, en una cancha de $120 \times 80$. Cada equipo ataca siempre hacia $x = 120$, así que $x > 90$ es el último cuarto de la cancha rival.
- `xg`: el gol esperado del remate (vacío si la acción no es un remate). `gol`: 1 si el remate fue gol.
- `peligro`: los puntos que **la receta del curso** le dio a esa acción.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# El partido de entrada. Para la tarea puente 2 solo cambia esta línea (ver el final del cuaderno).
RUTA_EVENTOS = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/eventos_final2022.csv"
ev = pd.read_csv(RUTA_EVENTOS)

LOCAL, VISITA = ev["equipo"].unique()[:2]     # el primero domina "hacia arriba" (+), el segundo "hacia abajo" (−)
print("Filas y columnas:", ev.shape)
print("Arriba (+):", LOCAL, "| Abajo (−):", VISITA)
ev.sample(8, random_state=1).sort_index()

In [ ]:
# ✏️ Ejercicio 1 — El ritual: mirar antes de calcular. Respondan con código.
print("Tipos de evento distintos:", ev["tipo"].nunique())
# print(ev["tipo"].value_counts())                    # a) ¿cuántos eventos hay de cada tipo?
# print(ev[["x", "y"]].describe())                    # b) ¿qué valores toman x e y? ¿respetan la cancha de 120 × 80?
# print(ev[ev["x"].isna()]["tipo"].value_counts())    # c) ¿qué eventos NO tienen coordenadas? ¿tiene sentido?
# print(ev.groupby("equipo")["tipo"].count())         # d) ¿quién hizo más acciones?

# 🗣️ Tres cuartas partes del archivo son pases, recepciones y conducciones. ¿Todas deberían contar como
#    "peligro"? Anoten su primera intuición: en la sección 3 la convierten en una decisión.

In [ ]:
# Los goles: los remates con gol == 1. La tanda de penales no está en el archivo (tarea puente 1).
goles = ev[ev["gol"] == 1][["t_corrido", "minuto", "equipo", "x", "y", "xg", "peligro"]].copy()
goles["minuto_corrido"] = goles["t_corrido"].astype(int)
print(goles.to_string(index=False))

# 🗣️ Confronten con la tarea puente 1: ¿cuáles tres goles tienen el mismo xG? ¿Desde qué punto (x, y) se patearon?
#    ¿Cuál es el gol con menor xG y cuántos puntos le dio la receta del curso?

---
## 2 · La receta del curso, auditada

Una receta de momentum tiene cuatro pasos, y la del curso está escrita al inicio de `preparar_momentum.py`:

1. **Puntos.** Cada acción recibe puntos de peligro según su tipo y su zona. Ese paso ya viene hecho en la columna `peligro`.
2. **Suma.** Se suman los puntos de cada equipo en cada minuto corrido $k$ (un `groupby`).
3. **Resta.** $r_k = \text{peligro}_{\text{local}}(k) - \text{peligro}_{\text{visita}}(k)$: el momentum bruto.
4. **Memoria.** Se suaviza con una media móvil exponencial de memoria $\tau$ minutos:
$$m_k = (1-\lambda)\,m_{k-1} + \lambda\, r_k, \qquad \lambda = \frac{1}{\tau}, \qquad m_{-1} = 0 .$$

El resultado es la escalonada $m(t)$ de la sesión 1. Las dos funciones de abajo hacen los pasos 2, 3 y 4 para **cualquier** columna de puntos, de modo que sirven igual para la receta del curso y para la suya. Léanlas línea por línea: el monitor puede pedirles que expliquen cualquiera.

In [ ]:
T = int(ev["t_corrido"].max()) + 1                         # minutos corridos del partido
ev["k"] = np.minimum(ev["t_corrido"].astype(int), T - 1)   # el minuto corrido (entero) de cada acción
t = np.arange(T + 1)                                       # la partición: 0, 1, ..., T
CORTES = [0] + [int(round(ev[ev["periodo"] == p]["t_corrido"].min())) for p in sorted(ev["periodo"].unique())[1:]] + [T]

def suavizar(bruto, memoria):
    # Paso 4: media móvil exponencial. memoria = τ en minutos; λ = 1/τ.
    lam = 1.0 / memoria
    m = np.zeros(len(bruto))
    for k in range(len(bruto)):
        anterior = m[k - 1] if k > 0 else 0.0
        m[k] = (1 - lam) * anterior + lam * bruto[k]
    return m

def momentum_desde_puntos(ev, columna, memoria):
    # Pasos 2, 3 y 4: de una columna de puntos por acción a la escalonada m_0, ..., m_{T-1}.
    por_minuto = ev.groupby(["k", "equipo"])[columna].sum().unstack(fill_value=0.0)   # paso 2
    por_minuto = por_minuto.reindex(index=range(T), columns=[LOCAL, VISITA], fill_value=0.0)
    bruto = (por_minuto[LOCAL] - por_minuto[VISITA]).to_numpy()                       # paso 3
    return suavizar(bruto, memoria)                                                   # paso 4

m_curso = momentum_desde_puntos(ev, "peligro", memoria=3)
D_curso = np.concatenate([[0.0], np.cumsum(m_curso)])      # D(x) = ∫_0^x m, como en la sesión 1
print("Minutos:", T, "| cortes de los tiempos:", CORTES)
print(f"Dominio neto con la receta del curso: {D_curso[-1]:+.2f} puntos de peligro")

# La auditoría: ¿esto reproduce el CSV congelado de la sesión 1? (Solo existe para la final 2022.)
if "final2022" in RUTA_EVENTOS:
    congelado = pd.read_csv(RUTA_EVENTOS.replace("eventos_", "momentum_"))["momentum"].to_numpy()
    print("Diferencia máxima con momentum_final2022.csv:", np.abs(m_curso - congelado).max())

In [ ]:
# ✏️ Ejercicio 2 — ¿Qué hace la memoria? Los mismos puntos del curso con tres memorias distintas.
fig, ax = plt.subplots(figsize=(12, 4.5))
for memoria, color in [(1, "#BBBBBB"), (3, "black"), (10, "#2F6DB5")]:
    m_tau = momentum_desde_puntos(ev, "peligro", memoria)
    ax.step(t, np.append(m_tau, m_tau[-1]), where="post", color=color, linewidth=1.2, label=f"memoria τ = {memoria} min")
    print(f"τ = {memoria:2d}:  dominio neto = {m_tau.sum():+.2f}   máximo de m = {m_tau.max():.2f} en el minuto {int(np.argmax(m_tau))}")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Minuto corrido"); ax.set_ylabel("m(t)  (puntos de peligro / min)")
ax.set_title("La misma materia prima con tres memorias"); ax.legend(); plt.show()

# 🗣️ Con τ = 1 no hay memoria: m es el momentum bruto. ¿Qué le hace la memoria a la altura de los picos?
#    ¿Y a su posición? Miren el dominio neto en los tres casos: cambia poco. ¿Por qué? (Pista: la memoria
#    reparte los puntos de un minuto entre los minutos siguientes. ¿Los crea o los destruye?)

---
## 3 · Diseñar la receta propia

Ahora deciden ustedes. Antes de escribir código, una advertencia sobre la materia prima: este archivo es una versión **simplificada** del crudo de StatsBomb. Trae dónde empieza cada acción, pero no dónde termina ni si salió bien (un pase completo y uno interceptado se ven igual). La receta del curso se calculó sobre el crudo completo, así que la suya no puede copiarla exactamente. Tiene que ser otra, diseñada con lo que hay en la tabla: `tipo`, `x`, `y`, `xg`, `gol`.

🗣️ **Las cuatro decisiones (doble clic para editar). Cada una con una línea de justificación:**

| Decisión | Lo que eligieron | ¿Por qué? |
|---|---|---|
| ¿Qué eventos cuentan como peligro? | | |
| ¿Cuánto pesa cada uno? | | |
| ¿Dónde empieza la zona de peligro? | | |
| ¿Cuánta memoria tiene el dominio? | | |

Pregunta guía para los pesos: ¿su receta le da más puntos a un gol que a un remate fallado con el mismo xG? La del curso sí (+2). Decidan si eso mide dominio o mide marcador.

In [ ]:
# ✏️ Ejercicio 3 — La receta del equipo. Lo que hay abajo es una SEMILLA para que el cuaderno corra:
# solo cuenta los remates y todos pesan lo mismo. Reemplácenla por la suya.
PESOS = {
    "remate": 1.0,          # puntos por cada remate
    # "xg": ...,            # ← ¿puntos adicionales según el xG del remate?
    # "gol": ...,           # ← ¿puntos adicionales si fue gol?
    # "pase_zona": ...,     # ← ¿puntos por un pase que sale desde la zona de peligro?
    # "regate": ...,        # ← ¿y los regates, las faltas recibidas, las conducciones?
}
ZONA_X = 90                 # la zona de peligro empieza en x = ZONA_X (90 es el último cuarto)
MEMORIA = 3                 # τ, en minutos

def mis_puntos(fila, pesos=PESOS):
    # Recibe una fila de la tabla de eventos y devuelve los puntos de peligro que le da SU receta.
    p = 0.0
    if fila["tipo"] == "Shot":
        p = p + pesos["remate"]
        # p = p + pesos["xg"] * fila["xg"]              # ← ejemplo de cómo usar el xG
    # elif fila["tipo"] == "Pass" and fila["x"] >= ZONA_X:
    #     p = p + pesos["pase_zona"]                    # ← ejemplo de cómo usar la zona
    return p

# Prueba rápida: ¿cuántos puntos le da su receta a cada uno de los goles?
for i in goles.index:
    print(f"gol en el minuto corrido {goles.loc[i, 'minuto_corrido']:3d} ({goles.loc[i, 'equipo']}): {mis_puntos(ev.loc[i]):.2f} puntos")

---
## 4 · Implementarla

Paso 1 con su receta: una columna nueva, `mis_puntos`, con los puntos de cada acción. Los pasos 2, 3 y 4 ya están escritos en `momentum_desde_puntos`. El resultado es otra función escalonada, su $m(t)$.

In [ ]:
ev["mis_puntos"] = ev.apply(mis_puntos, axis=1)                  # paso 1: puntos por acción

# ¿De dónde salen los puntos de su receta? Una tabla que deben poder defender.
reparto = ev[ev["mis_puntos"] > 0].groupby(["tipo", "equipo"])["mis_puntos"].agg(["count", "sum"]).round(2)
print(reparto)
print("\nTotal de puntos por equipo:\n", ev.groupby("equipo")["mis_puntos"].sum().round(2))

m_mia = momentum_desde_puntos(ev, "mis_puntos", MEMORIA)         # pasos 2, 3 y 4
D_mia = np.concatenate([[0.0], np.cumsum(m_mia)])

m_esc = np.append(m_mia, m_mia[-1])
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(t, np.maximum(m_esc, 0), 0, step="post", color="#2F6DB5", alpha=0.6, label=f"domina {LOCAL} (+)")
ax.fill_between(t, np.minimum(m_esc, 0), 0, step="post", color="#C9573C", alpha=0.6, label=f"domina {VISITA} (−)")
ax.step(t, m_esc, where="post", color="black", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.8)
for c in CORTES[1:-1]:
    ax.axvline(c, color="gray", linestyle="--")
ax.set_xlabel("Minuto corrido"); ax.set_ylabel("m(t)  (puntos de SU receta / min)")
ax.set_title("El momentum del equipo: otra función escalonada"); ax.legend(); plt.show()

In [ ]:
# ✏️ Ejercicio 4 — El paso 2 a mano, para un solo minuto.
# Con un filtro y un groupby, calculen cuántos puntos de SU receta generó cada equipo en el minuto corrido 125
# y la resta local − visita. Ese número es el momentum bruto r_125 de su receta.
# minuto_125 = ev[ev["k"] == 125]
# print(minuto_125.groupby("equipo")["mis_puntos"].sum())

# 🗣️ Las unidades: los "puntos" de su receta y los del curso no son la misma moneda. Si su m llega a 0,5 y
#    la del curso a 3,8, ¿eso dice que su receta ve menos dominio? ¿Qué sí se puede comparar entre las dos?

---
## 5 · Comparar contra la del curso

Dos recetas, el mismo partido. Como las unidades son distintas, la primera gráfica dibuja cada momentum dividido por su máximo valor absoluto, para comparar la **forma** y no la altura. La **correlación** resume esa comparación en un número entre $-1$ y $1$: cerca de $1$, las dos recetas suben y bajan en los mismos minutos.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True)
ax1.step(t, np.append(m_curso, m_curso[-1]) / np.abs(m_curso).max(), where="post", color="black", linewidth=1.2, label="receta del curso")
ax1.step(t, np.append(m_mia, m_mia[-1]) / np.abs(m_mia).max(), where="post", color="#E08A00", linewidth=1.2, label="receta del equipo")
ax1.axhline(0, color="black", linewidth=0.8); ax1.set_ylabel("m(t) / máx |m|")
ax1.set_title("Arriba las dos tasas (normalizadas), abajo las dos acumuladas (normalizadas)"); ax1.legend()
ax2.plot(t, D_curso / np.abs(D_curso).max(), color="black", linewidth=2, label="D del curso")
ax2.plot(t, D_mia / np.abs(D_mia).max(), color="#E08A00", linewidth=2, label="D del equipo")
ax2.axhline(0, color="black", linewidth=0.8); ax2.set_ylabel("D(x) / máx |D|"); ax2.set_xlabel("Minuto corrido"); ax2.legend()
for ax in (ax1, ax2):
    for c in CORTES[1:-1]:
        ax.axvline(c, color="gray", linestyle="--")
plt.show()

print(f"Correlación entre las dos tasas m: {np.corrcoef(m_curso, m_mia)[0, 1]:.3f}")

In [ ]:
def resumen(nombre, m):
    # Las preguntas de la sesión 1, para cualquier receta.
    D = np.concatenate([[0.0], np.cumsum(m)])
    print(f"{nombre:20s} dominio neto {D[-1]:+8.2f} | máx de m en el minuto {int(np.argmax(m)):3d} | máx de D en x = {int(np.argmax(D)):3d}")
    return D

_ = resumen("Receta del curso:", m_curso)
_ = resumen("Receta del equipo:", m_mia)

# ✏️ Ejercicio 5 — Los picos gemelos y P4.
# a) En la sesión 1, D tenía dos picos casi iguales: uno en el tiempo reglamentario y otro en la prórroga.
#    Encuéntrenlos en SU D (recuerden: np.argmax(D_mia[:CORTES[2] + 1]) busca solo en el tiempo reglamentario).
# b) P4: multipliquen por 10 el peso de los remates y recalculen, sin tocar la receta original.
# PESOS_X10 = dict(PESOS); PESOS_X10["remate"] = 10 * PESOS["remate"]
# ev["puntos_x10"] = ev.apply(mis_puntos, axis=1, pesos=PESOS_X10)
# _ = resumen("Remates × 10:", momentum_desde_puntos(ev, "puntos_x10", MEMORIA))

# 🗣️ ¿Su receta cuenta la misma historia que la del curso? ¿Cambió el signo del dominio neto? ¿Cambió el
#    minuto más ganado? ¿Siguen los picos gemelos? Confronten P4: ¿qué cambió al multiplicar por 10, el
#    signo, el máximo de D, o ninguno? Si su receta solo tiene remates, ¿por qué el resultado era previsible?
#    (Pista: ¿qué le pasa a una integral cuando el integrando se multiplica por una constante?)

---
## 6 · ¿Anticipa o celebra?

P5 con evidencia. Para cada gol se calcula el valor promedio de $m$ en los cinco minutos **anteriores**, el valor de $m$ en el minuto del gol y el promedio en los cinco minutos **posteriores**. Todo se mide *a favor del equipo que anotó*: si el gol es del equipo de abajo, se cambia el signo. Un número positivo antes del gol dice que el goleador venía dominando.

El valor promedio es el de la sesión 1: $\dfrac{1}{b-a}\displaystyle\int_a^b m(t)\,dt$, con $b - a = 5$.

In [ ]:
def antes_y_despues(m, ventana=5):
    filas = []
    for _, g in goles.iterrows():
        k = int(g["minuto_corrido"])
        signo = 1 if g["equipo"] == LOCAL else -1
        antes   = signo * m[max(k - ventana, 0):k].mean()        # (1/5) ∫ de m en [k−5, k]
        durante = signo * m[k]
        despues = signo * m[k + 1:k + 1 + ventana].mean()        # (1/5) ∫ de m en [k+1, k+6]
        filas.append((k, g["equipo"], round(g["xg"], 3), round(antes, 2), round(durante, 2), round(despues, 2)))
    return pd.DataFrame(filas, columns=["minuto", "anota", "xg", "antes", "en el gol", "después"])

print("Receta del curso (a favor del que anota):")
print(antes_y_despues(m_curso).to_string(index=False))

# ✏️ Ejercicio 6 — La misma tabla con SU receta.
# print(antes_y_despues(m_mia).to_string(index=False))

# 🗣️ a) ¿En cuáles goles el que anotó venía dominando (antes > 0) y en cuáles el gol llegó contra la corriente?
#    b) Miren los goles de Francia. ¿Qué hay dentro de los cinco minutos anteriores al segundo?
#    c) La columna "después" casi siempre es positiva. Antes de concluir que el gol "envalentona", piensen:
#       el gol mismo suma puntos, y la memoria los arrastra varios minutos. ¿Cómo separarían el efecto del gol
#       del dominio posterior? (Una idea: repetir la tabla con una receta en la que el gol no dé puntos extra.)
#    d) Veredicto sobre P5, con los números de la tabla y no con la impresión de la gráfica.

---
## 7 · La discusión final

Con la receta del curso, Argentina acumuló $+26{,}7$ puntos de dominio. El partido terminó 3–3 y se decidió en penales. **La integral mide dominio, no goles.**

🗣️ Discutan en el equipo y escriban aquí: ¿qué pregunta responde exactamente $\int_0^{142} m(t)\,dt$ con su receta? ¿Qué pregunta parecida **no** responde? Si alguien les dijera "Argentina mereció ganar porque la integral es positiva", ¿qué le contestarían?

---
## 🗣️ Momento 3 — Explicar

Una página del equipo, aquí mismo:

1. La defensa de su receta (pesos, zona, memoria) y qué historia cuenta distinto a la del curso.
2. El veredicto sobre P5, con evidencia de la sección 6.
3. La reflexión "la integral mide dominio, no goles", escrita con sus palabras.

---
## 🤖 Protocolo de IA (recordatorio)

La IA es tutora y auditora, no autora. Intento propio antes de consultar; la IA no es la autoridad final; **todo uso va a la bitácora individual**. Está prohibido pedirle los pesos de su receta, la respuesta a P5 o las conclusiones del momento 3. La cacería de errores de esta guía es puntuable: pídanle "un algoritmo de momentum para un partido de fútbol a partir de eventos", sin mostrarle la guía, y audítenlo contra el suyo. Todo integrante debe poder explicar cualquier celda si el monitor la señala.

---
## 🎒 Tarea puente 2 — Otro partido, la misma receta

Su algoritmo ya existe y ya fue defendido. Elijan un partido de los datos abiertos de StatsBomb, generen su archivo de eventos con la celda de abajo, cambien `RUTA_EVENTOS` en la primera celda de código y ejecuten todo de nuevo (`Entorno de ejecución → Ejecutar todas`). El cuaderno debe correr de arriba a abajo **sin retocar la receta**. Después escriban tres líneas: ¿la curva cuenta el partido que ustedes recuerdan? Donde no coincida, ¿el problema es del recuerdo o de la receta?

In [ ]:
# Tarea puente 2: quiten los # y ejecuten, una línea a la vez.
# !wget -q https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/preparar_momentum.py
# !python preparar_momentum.py --listar 223 282          # Copa América 2024 (43 106: Mundial 2022; 55 282: Euro 2024)
# !python preparar_momentum.py --match-id 3943077        # el id del partido elegido, tomado de la lista
#
# Eso deja en Colab el archivo eventos_3943077.csv. Ahora, en la primera celda de código del cuaderno:
# RUTA_EVENTOS = "eventos_3943077.csv"